# 02 · Bronze → Silver

Neste notebook é realizado o processo de higienização, padronização e enriquecimento dos dados.
As tabelas da camada Bronze são mantidas inalteradas, e são geradas as tabelas da camada Silver com:
- Nomes de colunas traduzidos para o português;
- Tipagem rigorosa dos dados;
- Deduplicação com retenção do registro mais recente (`ingestion_datetime`);
- Tratamento de inconsistências de origem (deslocamento de colunas, ruídos e erros de escala).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_analytics"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

print(f"Lendo da Bronze: {bronze_schema}")
print(f"Gravando na Silver: {silver_schema}")

Lendo da Bronze: cinedata_analytics.bronze
Gravando na Silver: cinedata_analytics.silver


## 1. Informações dos Filmes (`silver.tb_info_filmes`)

Nesta etapa são aplicados os seguintes tratamentos:
- **Deduplicação:** Remoção de duplicidades por `id_filme`, preservando a versão com maior `ingestion_datetime`.
- **Padronização de Status:** Limpeza de ruídos e hífens, tradução dos termos para português e mapeamento de registros corrompidos como "Não Informado".
- **Datas Multi-Formato:** Utilização de `coalesce` para testar os padrões de data presentes na base (`yyyy-MM-dd`, `dd/MM/yyyy`, `MM-dd-yyyy`, etc.), derivando a coluna `ano_lancamento`.

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

window_info = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_info_dedup = (
    df_bronze_info
    .filter(F.col("id").isNotNull() & (F.trim(F.col("id")) != ""))
    .withColumn("rn", F.row_number().over(window_info))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

status_norm = F.lower(F.trim(F.regexp_replace(F.regexp_replace(F.col("status"), "[^a-zA-Z\\s]", " "), "\\s+", " ")))

df_info_status = df_info_dedup.withColumn(
    "status_filme",
    F.when(status_norm == "released", "Lançado")
    .when(status_norm.isin("post production", "postproduction"), "Pós-Produção")
    .when(status_norm.isin("in production", "inproduction"), "Em Produção")
    .when(status_norm == "planned", "Planejado")
    .when(status_norm == "rumored", "Rumores")
    .when(status_norm.isin("canceled", "cancelled"), "Cancelado")
    .otherwise("Não Informado")
)

col_data = F.trim(F.col("release_date"))
df_info_dates = df_info_status.withColumn(
    "data_lancamento",
    F.coalesce(
        F.try_to_date(col_data, "yyyy-MM-dd"),
        F.try_to_date(col_data, "dd/MM/yyyy"),
        F.try_to_date(col_data, "MM-dd-yyyy"),
        F.try_to_date(col_data, "MM/dd/yyyy"),
        F.try_to_date(col_data, "yyyy/MM/dd"),
        F.try_to_date(col_data, "dd-MM-yyyy")
    )
).withColumn("ano_lancamento", F.year(F.col("data_lancamento")).cast("int"))

df_silver_info = (
    df_info_dates
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.trim(F.col("title")).alias("titulo"),
        F.trim(F.col("original_title")).alias("titulo_original"),
        F.col("data_lancamento"),
        F.col("ano_lancamento"),
        F.expr("try_cast(runtime as int)").alias("duracao_minutos"),
        F.trim(F.col("original_language")).alias("idioma_original"),
        F.col("status_filme"),
        F.trim(F.col("overview")).alias("sinopse"),
        F.trim(F.col("tagline")).alias("frase_divulgacao")
    )
)

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

print(f"[OK] silver.tb_info_filmes gravada com {df_silver_info.count()} registros.")
display(df_silver_info.limit(5))

[OK] silver.tb_info_filmes gravada com 97879 registros.


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
1000004,Purple Beatz,Purple Beatz,2022-07-07,2022,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,2021,0,es,Lançado,null,null
1000079,Pourquoi tu souris ?,Pourquoi tu souris ?,2024-06-26,2024,0,fr,Em Produção,null,null
1000099,Chowdhury Rajbari,Chowdhury Rajbari,2017-12-08,2017,131,bn,Lançado,"Giaa and Arnab are a married couple, who visit Arnab's ancestral palace 'Chowdhury Rajbari' for a romantic getaway. Once there, Giaa experiences strange occurrences and memories of a past life. In her search for truth, Giaa comes across Dr. Oindrila, who assists her in her search. But trouble starts when they come across more questions, instead of answers.",null
1000110,Les Enfants rouges,Les Enfants rouges,2023-10-25,2023,0,fr,Lançado,null,null


## 2. Série Temporal da Cotação do Dólar (`silver.tb_cotacao_dolar`)

Como a API do Banco Central não fornece cotações em finais de semana e feriados:
- É construído um calendário contínuo diário entre as datas mínima e máxima do histórico.
- É aplicada a técnica de preenchimento *Forward Fill* (`last` via Window function), garantindo que dias sem cotação recebam o valor do último dia útil disponível.

In [0]:
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cotacao_diaria = (
    df_bronze_cotacao
    .withColumn("data", F.to_date(F.col("dataHoraCotacao")))
    .filter(F.col("data").isNotNull())
    .groupBy("data")
    .agg(F.last("cotacaoCompra", ignorenulls=True).alias("cotacao_compra"))
)

limites = df_cotacao_diaria.select(
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max")
).collect()[0]

data_min = limites["data_min"]
data_max = limites["data_max"]

df_calendario = spark.sql(f"""
    SELECT explode(sequence(DATE('{data_min}'), DATE('{data_max}'), INTERVAL 1 DAY)) AS data
""")

window_ffill = (
    Window.orderBy("data")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacao_diaria, on="data", how="left")
    .withColumn(
        "cotacao_compra_preenchida",
        F.last("cotacao_compra", ignorenulls=True).over(window_ffill)
    )
    .select(
        F.col("data"),
        F.round(F.col("cotacao_compra_preenchida"), 4).alias("cotacao_compra")
    )
)

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

print("silver.tb_cotacao_dolar criada com Forward Fill.")
display(df_silver_cotacao.orderBy("data"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver.tb_cotacao_dolar criada com Forward Fill.


data,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692


## 3. Dados Financeiros dos Filmes (`silver.tb_financeiro_filmes`)

Tratamento de métricas financeiras de orçamento e receita:
- Conversão de textos que indicam ausência de dados ("Unknown", "Não Informado") para `NULL`.
- Remoção de símbolos monetários (`$`, `USD`), vírgulas e expansão de sufixos numéricos (`K` para milhares e `M` para milhões).
- Tratamento de valores negativos ou zerados como `NULL`.
- Conversão cambial de USD para BRL utilizando a taxa de referência da tabela de cotação.
- Cálculo de Lucro e Margem de Lucro Percentual de forma segura, prevenindo divisão por zero.

In [0]:
df_bronze_fin = spark.table(f"{bronze_schema}.tb_movies_financials")

taxa_dolar = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data").desc())
    .select("cotacao_compra")
    .first()[0]
)
print(f"Taxa de Cotação de Referência aplicada: R$ {taxa_dolar:.4f}")

window_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_fin_dedup = (
    df_bronze_fin
    .filter(F.col("id").isNotNull() & (F.trim(F.col("id")) != ""))
    .withColumn("rn", F.row_number().over(window_fin))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

def limpar_moeda(col_name):
    c = F.trim(F.col(col_name))
    c_limpa = F.when(c.isin("Unknown", "Não Informado", "N/A", "None", ""), None).otherwise(c)
    eh_milhao = F.upper(c_limpa).endswith("M")
    eh_mil = F.upper(c_limpa).endswith("K")
    c_num = F.regexp_replace(c_limpa, "[\\$,]|(?i)USD|\\s|(?i)M|(?i)K", "")
    val_float = c_num.cast("double")
    val_ajustado = (
        F.when(eh_milhao, val_float * F.lit(1000000.0))
        .when(eh_mil, val_float * F.lit(1000.0))
        .otherwise(val_float)
    )
    return F.when(val_ajustado > 0, val_ajustado.cast("decimal(18,2)")).otherwise(None)

df_silver_fin = (
    df_fin_dedup
    .withColumn("orcamento_usd", limpar_moeda("budget"))
    .withColumn("receita_usd", limpar_moeda("revenue"))
    .withColumn("taxa_cotacao", F.lit(taxa_dolar))
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.col("taxa_cotacao"), 2).cast("decimal(18,2)"))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.col("taxa_cotacao"), 2).cast("decimal(18,2)"))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            (F.col("receita_usd") > 0) & F.col("lucro_usd").isNotNull(),
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)
        ).otherwise(None)
    )
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual"
    )
)

df_silver_fin.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"silver.tb_financeiro_filmes gravada com {df_silver_fin.count()} registros.")
display(df_silver_fin.filter(F.col("receita_usd").isNotNull()).limit(5))

Taxa de Cotação de Referência aplicada: R$ 4.9692
silver.tb_financeiro_filmes gravada com 99006 registros.


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1001724,3000.00,8000.00,14907.60,39753.60,5000.00,24846.00,62.50
1003578,10000.00,3000000.00,49692.00,14907600.00,2990000.00,14857908.00,99.67
1008903,null,3600000.00,null,17889120.00,null,null,null
1011301,null,4265.00,null,21193.64,null,null,null
1022145,null,236085820.00,null,1173157656.74,null,null,null


Aqui, por conta da grande quantidade de valores null, optei por verificar se o dataframe estava retornando corretamente os dados de filmes que possuiam a informação de orçamento e lucro.

In [0]:
display(
    df_silver_fin
    .filter(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull())
    .orderBy(F.col("receita_usd").desc())
    .limit(5)
)

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
299534,356000000.00,2800000000.00,1769035200.00,13913760000.00,2444000000.00,12144724800.00,87.29
76600,460000000.00,2320250281.00,2285832000.00,11529787696.35,1860250281.00,9243955696.35,80.17
299536,300000000.00,2052415039.00,1490760000.00,10198860811.80,1752415039.00,8708100811.80,85.38
634649,200000000.00,1921847111.00,993840000.00,9550042663.98,1721847111.00,8556202663.98,89.59
420818,260000000.00,1663075401.00,1291992000.00,8264154282.65,1403075401.00,6972162282.65,84.37


## 4. Métricas de Engajamento (`silver.tb_metricas_engajamento`)

Tratamento das notas e votos dos portais TMDB e IMDb:
- **Popularidade:** Correção de separadores decimais (vírgula para ponto) e expurgo de valores negativos.
- **Tipagem Segura:** Uso de `try_cast` para lidar com o deslocamento de colunas (*Column Shift*), evitando que textos espalhados em campos numéricos quebrem a execução.
- **Limites de Negócio:** Descarte de notas fora do intervalo válido de 0 a 10 (incluindo valores multiplicados por erro de digitação/escala), tratando-os como `NULL`.

In [0]:
df_bronze_met = spark.table(f"{bronze_schema}.tb_movies_metrics")

window_met = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_met_dedup = (
    df_bronze_met
    .filter(F.col("id").isNotNull() & (F.trim(F.col("id")) != ""))
    .withColumn("rn", F.row_number().over(window_met))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

pop_limpa = F.regexp_replace(F.regexp_replace(F.trim(F.col("popularity")), "\"", ""), ",", ".")
val_pop = F.expr("try_cast(regexp_replace(regexp_replace(trim(popularity), '\"', ''), ',', '.') as double)")

nota_tmdb_val = F.expr("try_cast(replace(trim(vote_average), ',', '.') as double)")
nota_imdb_val = F.expr("try_cast(replace(trim(averageRating), ',', '.') as double)")

votos_tmdb_val = F.expr("try_cast(trim(vote_count) as int)")
votos_imdb_val = F.expr("try_cast(trim(numVotes) as int)")

df_silver_met = (
    df_met_dedup
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.when(val_pop >= 0, val_pop).otherwise(None).alias("popularidade"),
        F.when((nota_tmdb_val >= 0.0) & (nota_tmdb_val <= 10.0), nota_tmdb_val).otherwise(None).alias("nota_media_tmdb"),
        F.when(votos_tmdb_val >= 0, votos_tmdb_val).otherwise(None).alias("qtd_votos_tmdb"),
        F.when((nota_imdb_val >= 0.0) & (nota_imdb_val <= 10.0), nota_imdb_val).otherwise(None).alias("nota_media_imdb"),
        F.when(votos_imdb_val >= 0, votos_imdb_val).otherwise(None).alias("qtd_votos_imdb")
    )
)

df_silver_met.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

print(f"silver.tb_metricas_engajamento gravada com {df_silver_met.count()} registros.")
display(df_silver_met.orderBy(F.col("popularidade").desc()).limit(5))

silver.tb_metricas_engajamento gravada com 99013 registros.


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
565770,2994.357,7.139,1023,5.9,119501
980489,2680.593,8.068,null,7.1,138696
800297,2020.0,10.0,1,null,null
809905,2020.0,5.0,2,null,null
658829,2019.0,5.4,5,null,null


## 5. Avaliações dos Usuários (`silver.tb_avaliacoes_usuarios`)

Processamento dos comentários e notas individuais dos usuários:
- Deduplicação integral de linhas com mesmo filme, usuário, nota e comentário idênticos.
- Validação da nota na faixa permitida de 0 a 10.
- Padronização de comentários vazios ou preenchidos apenas com espaços como "Sem comentário".

In [0]:
df_bronze_rev = spark.table(f"{bronze_schema}.tb_movies_reviews")

nota_usr_val = F.expr("try_cast(replace(trim(nota), ',', '.') as double)")

df_silver_rev = (
    df_bronze_rev
    .filter(F.col("id").isNotNull() & (F.trim(F.col("id")) != ""))
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.trim(F.col("nome")).alias("nome_usuario"),
        F.when((nota_usr_val >= 0.0) & (nota_usr_val <= 10.0), nota_usr_val).otherwise(None).alias("nota_usuario"),
        F.when(F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""), "Sem comentário")
         .otherwise(F.trim(F.col("comentario"))).alias("comentario_usuario")
    )
)

df_silver_rev.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

print(f"[OK] silver.tb_avaliacoes_usuarios gravada com {df_silver_rev.count()} avaliações.")
display(df_silver_rev.limit(5))

[OK] silver.tb_avaliacoes_usuarios gravada com 32412 avaliações.


id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.


## 6. Gêneros Cinematográficos (`silver.tb_generos`)

Extração e limpeza da lista de gêneros por filme:
- Unificação de separadores mistos (vírgula, ponto-e-vírgula e barra vertical).
- Desmembramento via `split` e `explode`, gerando uma linha por gênero/filme.
- Filtragem restrita ao domínio oficial de gêneros cinematográficos, eliminando ruídos de deslocamento de colunas (sinopses, números e nomes de arquivos de imagem).

In [0]:
df_bronze_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")

generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

generos_explodidos = (
    df_bronze_credits
    .filter(F.col("id").isNotNull() & F.col("genres").isNotNull())
    .withColumn("genres_norm", F.regexp_replace(F.col("genres"), "[;|]", ","))
    .withColumn("genero_bruto", F.explode(F.split(F.col("genres_norm"), ",")))
    .withColumn("nome_genero", F.initcap(F.trim(F.col("genero_bruto"))))
)

df_silver_generos = (
    generos_explodidos
    .filter(F.col("nome_genero").isin(generos_validos))
    .select(
        F.trim(F.col("id")).alias("id_filme"),
        F.col("nome_genero")
    )
    .dropDuplicates(["id_filme", "nome_genero"])
)

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

print(f"silver.tb_generos gravada com {df_silver_generos.count()} relações filme-gênero.")
display(df_silver_generos.limit(5))

silver.tb_generos gravada com 138147 relações filme-gênero.


id_filme,nome_genero
271110,Adventure
346364,Horror
321612,Family
260513,Action
343668,Comedy


## 7. Pessoas e Empresas (`silver.tb_pessoas_empresas`)

Consolidação de quatro tipos de entidades em uma dimensão unificada:
- Ator (`cast`), Diretor (`directors`), Roteirista (`writers`) e Produtora (`production_companies`).
- Desmembramento das listas por participante, padronização de capitalização com `initcap` e eliminação de registros nulos ou espúrios.

In [0]:
def processar_entidade(col_origem, tipo_entidade):
    return (
        df_bronze_credits
        .filter(F.col("id").isNotNull() & F.col(col_origem).isNotNull())
        .withColumn("col_norm", F.regexp_replace(F.col(col_origem), ";", ","))
        .withColumn("item", F.explode(F.split(F.col("col_norm"), ",")))
        .withColumn("nome_entidade", F.initcap(F.trim(F.col("item"))))
        .filter(
            (F.col("nome_entidade") != "") &
            (~F.col("nome_entidade").isin("N/A", "None", "Nenhum", "Null")) &
            (~F.col("nome_entidade").rlike("^[0-9]")) &
            (F.length(F.col("nome_entidade")) > 1)
        )
        .select(
            F.trim(F.col("id")).alias("id_filme"),
            F.col("nome_entidade"),
            F.lit(tipo_entidade).alias("tipo_entidade")
        )
    )

df_atores = processar_entidade("cast", "Ator")
df_diretores = processar_entidade("directors", "Diretor")
df_roteiristas = processar_entidade("writers", "Roteirista")
df_produtoras = processar_entidade("production_companies", "Produtora")

df_silver_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

print(f"silver.tb_pessoas_empresas gravada com {df_silver_pessoas_empresas.count()} registros unificados.")
display(df_silver_pessoas_empresas.limit(5))

silver.tb_pessoas_empresas gravada com 907874 registros unificados.


id_filme,nome_entidade,tipo_entidade
381288,Izzie Coffey,Ator
392044,Michelle Pfeiffer,Ator
464052,Kristoffer Polaha,Ator
278927,Ben Kingsley,Ator
188927,Zachary Quinto,Ator
